# Recommender Systems - Part A Project
**Paper:** SVD-GoRank: Recommender System Algorithm Using SVD and Gower's Ranking

**Dataset:** Epinions (Product Reviews)

**Team Members:** Harun Korkmaz, Muhammet Salih Hasılcıo, Orhan Efe Bayrak, Muhiddin Fırat, Yasin Furkan Abasız, Yakup Berkay Genceroğlu, Görkem Yahya Bakan

This notebook contains data preprocessing and matrix construction steps for the Epinions-derived product review dataset. With over 250,000 ratings, it is critical for testing SVD scalability.

# ==========================================
# PROGRESS 1: DATA LOADING & PREPROCESSING (.mat)
# ==========================================

In [1]:
import pandas as pd
import numpy as np
import scipy.io as sio
from sklearn.model_selection import train_test_split


file_path_epi = r"C:\Users\Harun\Desktop\üni\4.SINIF\2.donem\recommender\dataset\epinions\rating.mat"
mat_data_epi = sio.loadmat(file_path_epi)

# ERROR FIX: Data has 4 columns, so we reduced the list to 4 elements
columns_tang = ['user_id', 'item_id', 'category_id', 'rating']
df_epi = pd.DataFrame(mat_data_epi['rating'], columns=columns_tang)

# Convert IDs to integers
df_epi['user_id'] = df_epi['user_id'].astype(int)
df_epi['item_id'] = df_epi['item_id'].astype(int)

# Only User, Item and Rating are needed. Drop others.
df_epi = df_epi[['user_id', 'item_id', 'rating']]

# Calculate statistics
n_users_epi = df_epi['user_id'].nunique()
n_items_epi = df_epi['item_id'].nunique()
n_ratings_epi = len(df_epi)
sparsity_epi = 1.0 - (n_ratings_epi / (n_users_epi * n_items_epi))

print("--- EPINIONS - PROGRESS 1 ---")
print(f"Total Users: {n_users_epi}")
print(f"Total Products: {n_items_epi}")
print(f"Total Ratings: {n_ratings_epi}")
print(f"Sparsity Ratio: {sparsity_epi:.6f} ({sparsity_epi*100:.4f}%)")
print("\nFirst 5 Rows of Simplified Dataset:")
display(df_epi.head())

--- EPINIONS - PROGRESS 1 ---
Total Users: 22164
Total Products: 296277
Total Ratings: 922267
Sparsity Ratio: 0.999860 (99.9860%)

First 5 Rows of Simplified Dataset:


,user_id,item_id,rating
0,1,1,2
1,1,2,2
2,1,3,2
3,1,4,5
4,1,5,3


# ==========================================
# PROGRESS 2: TRAIN/TEST SPLIT & MATRIX
# ==========================================

In [2]:
train_epi, test_epi = train_test_split(df_epi, test_size=0.20, random_state=42)

print("\n--- EPINIONS - PROGRESS 2 ---")
print(f"Training Set Size: {len(train_epi)} rows")
print(f"Test Set Size: {len(test_epi)} rows\n")

print("CRITICAL ENGINEERING NOTE")
print("Epinions matrix is huge, so to prevent MemoryError risk,")
print("we create a sample matrix with only top 500 users and 500 products.\n")

# To show the professor, create downsampled example data
top_users_epi = train_epi['user_id'].value_counts().index[:500]
top_items_epi = train_epi['item_id'].value_counts().index[:500]
sample_df_epi = train_epi[train_epi['user_id'].isin(top_users_epi) & train_epi['item_id'].isin(top_items_epi)]

# ERROR FIX 2: Use pivot_table to avoid duplicate record crashes like in Ciao
sample_matrix_epi = sample_df_epi.pivot_table(
    index='user_id', 
    columns='item_id', 
    values='rating',
    aggfunc='mean'
)

print(f"Sample Matrix Size: {sample_matrix_epi.shape}")
print("Sample Matrix Visualization (Empty cells are NaN):")
display(sample_matrix_epi.head())


--- EPINIONS - PROGRESS 2 ---
Training Set Size: 737813 rows
Test Set Size: 184454 rows

CRITICAL ENGINEERING NOTE
Epinions matrix is huge, so to prevent MemoryError risk,
we create a sample matrix with only top 500 users and 500 products.

Sample Matrix Size: (454, 499)
Sample Matrix Visualization (Empty cells are NaN):


item_id,1,2,7,9,18,20,26,28,29,32,...,17883,18431,18562,18666,19024,19449,23840,23962,25770,27073
user_id,,,,,,,,,,,,,,,,,,,,,
43,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
61,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
64,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
266,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
460,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
